In [22]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "tempelmann2013apes")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Results_conspecific_comprehension_study.sav")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [23]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_spss(complete_path_1, usecols=None, convert_categoricals=True)
# original_data_pathway_out = os.path.join(original_data_pathway, 'Results_conspecific_comprehension_study.csv')
# df.to_csv(original_data_pathway_out, encoding='utf-8-sig', index=False)
df['study_id']="tempelmann2013apes"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
# df.columns

In [24]:
df.rename(columns={"individual": "participant",
                   "species":"subgroup_temp",
                   'sex':'sex_original'}, inplace=True)
df[['subgroup', 'subgroup_temp2']] = df['subgroup_temp'].str.split('-',expand=True)
df['subgroup'].replace(['orang','bonobo'], np.nan, inplace=True)


In [25]:
intentionalt1_randomt1_list = [['weiter becher', 'far_cup'],
                    ['naher becher','near_cup'],
                    ['mittlerer becher','middle_cup']]
for x,y in intentionalt1_randomt1_list:
    df['intentionalt1'].replace(x, y, inplace=True, regex=True)
    df['randomt1'].replace(x, y, inplace=True, regex=True)

intcorrectt1_rand_correctt1_list = [['nicht korrekt','incorrect'],
                       ['korrekt','correct']]
for x,y in intcorrectt1_rand_correctt1_list:
    df['intcorrectt1'].replace(x, y, inplace=True, regex=True)
    df['rand_correctt1'].replace(x, y, inplace=True, regex=True)

In [26]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")
df_name  = pd.read_csv(comp_path_name_errors)
df['participant'] = df['participant'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['participant'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df= df.merge(apedf,left_on='participant', right_on='name', how='left')

In [27]:
df['condition_int']="intentional_communication"
df['condition_ran']="non_communicative"

In [28]:
# df.columns

df_temp = df[['study_id','participant','age','sex', 'species', 
       'subgroup','rear_hist', 'condition_int', 
       'condition1st', 'intentionalt1', 'target_post1', 'intcorrectt1',
       'int_overall', 'int_corr_1to6', 'int_corr_13to18', 'int_corr_1to9',
       'int_corr_10to18', 'position_1_int', 'position_2_int', 'position_3_int']].values.tolist() + df[['study_id',
       'participant','age','sex', 'species', 
       'subgroup','rear_hist','condition_ran',
       'condition1st','randomt1', 'target_pos_randt1', 'rand_correctt1', 
       'rand_overall','rand_corr_1to6', 'rand_corr_13to18', 'rand_corr_1to9',
       'rand_corr_10to18', 'position_1_rand', 'position_2_rand','position_3_random' ]].values.tolist()

df1 = pd.DataFrame(df_temp, columns=['study_id','participant','age','sex', 'species', 
       'subgroup','rear_hist','condition',
       'condition1st', 'location_signaled_by_participant', 'food_location', 'correct_trial_1',
       'average_overall_correct', 'average_correct_1-6', 'average_correct_13-18', 'average_correct_1-9',
       'average_correct_10-18', 'point_to_location_1', 'point_to_location_2', 'point_to_location_3'])
# df1.columns

In [29]:

food_list = [[1, 'far_cup'],
                    [3,'near_cup'],
                    [2,'middle_cup']]
for x,y in food_list:
    df1['food_location'].replace(x, y, inplace=True, regex=True)

In [30]:
df1.rename(columns={"rear_hist": "rearing"}, inplace=True)
space_list = ['rearing','condition1st', ]
for x in space_list:
    df1[x].replace(' ', '_', inplace=True, regex=True)

df1.rename(columns={"age":"age_in_years", "subgroup":"species_subgroup", "condition1st":"first_condition"}, inplace=True)

In [31]:
df1['first_condition'].replace('random', 'non_communicative', inplace=True, regex=True)
df1['first_condition'].replace('_first', '', inplace=True, regex=True)
df1['first_condition'].replace('intentional', 'intentional_communication', inplace=True, regex=True)

In [32]:
studyID_standardized=df1[['study_id', 'participant', 'age_in_years', 'sex', 'species', 'species_subgroup',
       'condition', 'first_condition',
       'location_signaled_by_participant', 'food_location', 'correct_trial_1',
       'average_overall_correct', 'average_correct_1-6',
       'average_correct_13-18', 'average_correct_1-9', 'average_correct_10-18',
       'point_to_location_1', 'point_to_location_2', 'point_to_location_3']]
comp_out_path_stand = os.path.join(out_pathway, 'tempelmann2013apes_standardized.csv')
studyID_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =studyID_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
studyID_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'tempelmann2013apes_glossary.csv')
studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)